# 🚀 生成式AI應用開發｜第03週
## Gemini API 入門：從 API Key 到第一個 Python AI 函式

> **執行環境：Google Colab** ｜ **本版本使用 Gemini API**

本教材內容對應 OpenAI API 入門版，但改用 Google Gemini API。重點是讓你比較不同 LLM 平台在 Python SDK、API key、輸入格式、回應物件與 **usage** 欄位上的差異。

官方文件入口：
- Gemini API Get started: https://ai.google.dev/gemini-api/docs/get-started
- Gemini text generation: https://ai.google.dev/gemini-api/docs/text-generation

> **本週任務**
> 完成第一個 API 呼叫，並把呼叫流程封裝成可重複使用的 Python 函式。


> **教師版**：本檔保留完整參考答案，可用於課堂示範、投影講解或課後核對。

## 🎯 本週學習目標

| # | 能力 | Gemini 對應概念 |
|---|------|----------------|
| 1 | 安裝與匯入 Gemini SDK | `google-genai` |
| 2 | 使用 Colab Secrets 讀取 API key | `GEMINI_API_KEY` |
| 3 | 建立 Gemini client | `genai.Client()` |
| 4 | 呼叫 Interactions API | `client.interactions.create()` |
| 5 | 使用 `system_instruction` 與 `input` | 角色設定與使用者輸入 |
| 6 | 解析 `interaction.output_text` 與 **usage** | 顯示回答、除錯、估算成本 |
| 7 | 封裝函式與錯誤處理 | 後續 App 開發 |


---
## 🧰 第一節：安裝套件

In [ ]:
!pip install -U google-genai --quiet
print("Google GenAI SDK 安裝完成")

In [ ]:
import os
import json
from google import genai

print("模組載入完成")

---
## 🔐 第二節：設定 API Key

請到 Google AI Studio 建立 API key，並在 Colab Secrets 新增：

- Name: `GEMINI_API_KEY`
- Value: 你的 Gemini API key
- 開啟 Notebook access

**不要把真正的 API key** 寫在 notebook 裡。

> **安全提醒**
> **API key 是機密資訊**。請使用 Colab Secrets 或環境變數，不要直接寫在 notebook、作業或截圖中。


In [ ]:
# Colab 優先從 Secrets 讀取 API key；本機執行時改用環境變數。
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        # 只把 key 放入環境變數，不要直接寫在程式碼或 notebook 輸出中。
        os.environ["GEMINI_API_KEY"] = api_key
        print(f"已讀取 Gemini API key（前 8 碼）：{api_key[:8]}...")
    else:
        print("Secrets 中找不到 GEMINI_API_KEY，請先完成設定")
except Exception as e:
    print(f"目前可能不是在 Colab 執行：{e}")
    print("若在本機執行，請先設定環境變數 GEMINI_API_KEY")

# 只顯示是否已設定，避免把完整 API key 印出來。
print("目前讀取狀態：", "已設定" if os.getenv("GEMINI_API_KEY") else "未設定")


### 💻 本機環境變數設定方式

```powershell
$env:GEMINI_API_KEY="你的 Gemini API key"
setx GEMINI_API_KEY "你的 Gemini API key"
```

```bash
export GEMINI_API_KEY="你的 Gemini API key"
```

---
## 🔌 第三節：建立 Gemini Client

`genai.Client()` 會從環境變數讀取 `GEMINI_API_KEY`。

In [ ]:
client = genai.Client()
# client 會自動從 GEMINI_API_KEY 環境變數讀取金鑰。

# 課堂預設使用 Gemini Flash。若老師課前確認其他模型更適合，可用 GEMINI_MODEL 覆蓋。
# 模型名稱集中在 MODEL，後續所有範例都共用同一設定。
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

print(f"Client 建立完成，使用模型：{MODEL}")


---
## ✨ 第四節：第一個 Gemini API 呼叫

Gemini Interactions API 使用 `client.interactions.create()`，並以 `interaction.output_text` 取出文字輸出。

In [ ]:
interaction = client.interactions.create(
    model=MODEL,
    input="請用繁體中文，用三句話解釋什麼是生成式 AI。"
)

# 這是最小可執行的單次 API 呼叫，用來確認 client、model 與 API key 都正常。
print(interaction.output_text)


### 🔍 4-1 檢查 interaction 物件

實務開發時，不只要看回答，也要觀察 id、model、**usage** 等欄位。


In [ ]:
print("interaction id:", interaction.id)
print("model:", interaction.model)
print("output_text:")
print(interaction.output_text)

print()
print("usage:")
# usage 可用來觀察 token 用量與後續成本估算。
print(interaction.usage)


---
## 🧭 第五節：加入 system_instruction

Gemini 使用 `system_instruction` 設定模型行為，概念上類似 OpenAI 版本的 `instructions`。它適合放角色、語氣、格式限制與穩定規則。

In [ ]:
# system_instruction 負責設定角色與規則，input 放使用者這一輪的問題。
interaction = client.interactions.create(
    model=MODEL,
    system_instruction="你是一位資管系 Python 助教。回答要精簡、具體，並使用繁體中文。",
    input="請解釋 API key 為什麼不能直接寫在程式碼裡。"
)

print(interaction.output_text)


---
## 💬 第六節：多輪對話與 previous_interaction_id

Gemini Interactions API 的多輪對話主線是使用 `previous_interaction_id`，讓伺服器延續前一次 interaction 的上下文。

這和 OpenAI / Claude 常見的 messages list 不同：

| 平台 | 多輪對話常見做法 |
|---|---|
| OpenAI Responses API | 可用 role/content list 或 previous response |
| Gemini Interactions API | 使用 `previous_interaction_id` 延續對話 |
| Claude Messages API | 每次送出完整 messages history |


In [ ]:
# Gemini 用 previous_interaction_id 串接上一輪 interaction。
interaction1 = client.interactions.create(
    model=MODEL,
    input="我正在學習 Python API 開發。"
)
print("第一輪：")
print(interaction1.output_text)

interaction2 = client.interactions.create(
    model=MODEL,
    input="根據上一句，請建議我下一步該練什麼。",
    # previous_interaction_id 讓第二輪沿用第一輪上下文。
    previous_interaction_id=interaction1.id
)
print()
print("第二輪：")
print(interaction2.output_text)


### 🧠 6-1 什麼時候使用 system_instruction？

1. 規則固定、每次都要遵守時，放在 `system_instruction`。
2. 使用者這次提出的問題，放在 `input`。
3. 要延續上一輪對話時，使用 `previous_interaction_id`。
4. 做聊天 App 時，需記錄上一輪 interaction id，或自行管理歷史。

---
## 🧩 第七節：封裝成函式

In [ ]:
def ask_ai(user_input, role="你是一位有幫助的助理", model=MODEL):
    # 把 Gemini API 呼叫包成函式，後面練習就不用重複寫 create()。
    # role 對應 system_instruction；user_input 對應 input。
    interaction = client.interactions.create(
        model=model,
        system_instruction=role,
        input=user_input
    )
    return interaction.output_text


answer = ask_ai("請用 3 個重點說明 Python 函式的用途。")
print(answer)


---
## 🛡️ 第八節：基本錯誤處理

In [ ]:
def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    # safe 版本把 API 例外轉成 (False, 錯誤訊息)，方便 notebook 繼續往下跑。
    try:
        interaction = client.interactions.create(
            model=model,
            system_instruction=role,
            input=user_input
        )
        return True, interaction.output_text
    except Exception as e:
        # 失敗時不要假裝模型有回答，要明確標示 API 呼叫失敗。
        return False, f"API 呼叫失敗：{e}"


success, result = ask_ai_safe("請用一句話說明 JSON 是什麼。")
print(result)


---
## 📊 第九節：讀取 **usage** 並建立除錯資訊


In [ ]:
def ask_ai_with_metadata(user_input, role="你是一位有幫助的助理", model=MODEL):
    # metadata 版本保留 id、model、answer、usage，方便比較與成本觀察。
    interaction = client.interactions.create(
        model=model,
        system_instruction=role,
        input=user_input
    )
    # 回傳 dict 比只回傳文字更適合後續自動化處理。
    return {
        "id": interaction.id,
        "model": interaction.model,
        "answer": interaction.output_text,
        "usage": interaction.usage,
    }


result = ask_ai_with_metadata("請列出學習 Gemini API 前需要會的 3 個 Python 技能。")
print(json.dumps(result, ensure_ascii=False, indent=2, default=str))


---
## 🧪 第十節：課堂練習

### 📝 練習 A：Python 作業批改助教

請讓 Gemini 扮演 Python 作業批改助教，檢查一段縮排錯誤的程式，並輸出「錯誤原因、修正方式、修正版程式碼」。

> **課堂練習**
> 先完成練習 A，再依時間進行練習 B 與選做練習 C。


In [ ]:
# 練習 A：把學生程式當成資料放入 prompt，請模型扮演批改助教。
student_code = '''
def add_numbers(a, b):
return a + b

print(add_numbers(3, 5))
'''

role = '''
你是一位 Python 作業批改助教。
請用繁體中文回答，並固定輸出三段：
1. 錯誤原因
2. 修正方式
3. 修正版程式碼
'''

# prompt 要清楚標明任務，再附上待檢查的程式碼。
prompt = f"請檢查以下 Python 程式：\n\n{student_code}"
success, result = ask_ai_safe(prompt, role=role)
print(result)


---
### 💰 練習 B：**usage** 與費用估算

請讀取 `usage`，練習把 token 使用量整理成 dict。價格會變動，正式估價前請查官方 pricing。

> **成本提醒**
> 本節重點是學會讀取 **usage** 與建立估算流程；實際價格請以上課當天官方 pricing 為準。


In [ ]:
def get_usage_value(usage, field_name):
    # usage 可能是 SDK 物件或 dict，因此用共用函式安全取值。
    if usage is None:
        return 0
    if isinstance(usage, dict):
        return usage.get(field_name, 0) or 0
    return getattr(usage, field_name, 0) or 0


def summarize_usage(usage):
    # Gemini 的 usage 欄位名稱和 OpenAI 不同，整理時要用平台自己的欄位。
    return {
        "total_input_tokens": get_usage_value(usage, "total_input_tokens"),
        "total_output_tokens": get_usage_value(usage, "total_output_tokens"),
        "total_tokens": get_usage_value(usage, "total_tokens"),
    }


result = ask_ai_with_metadata("請用兩句話說明為什麼 API 呼叫需要注意成本。")
print(result["answer"])
print(json.dumps(summarize_usage(result["usage"]), ensure_ascii=False, indent=2))


---
### 🎨 練習 C（選做）：自由設計一個角色

自訂至少兩個 `system_instruction`，觀察同一個問題在不同角色設定下的回答差異。

In [ ]:
# 練習 C：同一個問題搭配不同 role，觀察角色設定如何改變回答角度。
question = "我想做一個生成式 AI 期末專題，可以從哪裡開始？"
roles = [
    "你是一位資管系學習顧問，請用務實、可執行的方式回答。",
    "你是一位 AI 產品經理，請從使用者需求、功能設計、風險控管三個角度回答。",
]

for i, role in enumerate(roles, 1):
    print(f"===== 角色 {i} =====")
    success, result = ask_ai_safe(question, role=role)
    print(result)
    print()


---
## ✅ 本週小結

| 技能 | Gemini 寫法 |
|------|-------------|
| API key | `GEMINI_API_KEY` |
| Client | `client = genai.Client()` |
| 文字生成 | `client.interactions.create()` |
| 角色設定 | `system_instruction` |
| 文字輸出 | `interaction.output_text` |
| 多輪對話 | `previous_interaction_id` |
